# Pressure Visualization
Visualize upstream and downstream pressure over time from `cold_flow_data.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("cold_flow_data.csv")
df.columns = [str(c).strip() for c in df.columns]
df = df.rename(columns={"Unnamed: 2": "upstream_tag", "Unnamed: 4": "downstream_tag"})
df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
df["Upstream Pressure"] = pd.to_numeric(df["Upstream Pressure"], errors="coerce")
df["Downstream Pressure"] = pd.to_numeric(df["Downstream Pressure"], errors="coerce")
df = df[df["START"].eq("DATA")].dropna(subset=["Time", "Upstream Pressure", "Downstream Pressure"])
df.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure")
plt.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure")
plt.xlabel("Time")
plt.ylabel("Pressure")
plt.title("Upstream vs Downstream Pressure")
plt.legend()
plt.tight_layout()
plt.show()

## Savitzky-Golay Filter

Simulate a live savgol filter by taking the point in the middle of the filter window to avoid problematic behavior at the end of the curve if taking the last sample.

Different window lengths are explored with a 3rd order polynomial.

In [ ]:
from scipy.signal import savgol_filter

# Plot with multiple window lengths stacked vertically
window_lengths = [31, 51, 71]
polyorder = 3

if len(df) < max(window_lengths):
    raise ValueError("Not enough data points for the largest Savitzky-Golay window.")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, window_length in zip(axes, window_lengths):
    if window_length % 2 == 0:
        window_length -= 1
    if window_length < 5:
        raise ValueError("Window length is too small for Savitzky-Golay filtering.")

    half_window = window_length // 2

    upstream_filtered = savgol_filter(df["Upstream Pressure"], window_length, polyorder)
    downstream_filtered = savgol_filter(df["Downstream Pressure"], window_length, polyorder)

    # Shift forward by half_window to simulate live reporting at the window center
    shifted_upstream = pd.Series(upstream_filtered).shift(half_window)
    shifted_downstream = pd.Series(downstream_filtered).shift(half_window)

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], shifted_upstream, label=f"Upstream (Savgol Live, {window_length})")
    ax.plot(df["Time"], shifted_downstream, label=f"Downstream (Savgol Live, {window_length})")
    ax.set_ylabel("Pressure")
    ax.set_title(f"Savgol Live Window Length {window_length}")
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()

## Exponential Moving Average

In [ ]:
ema_spans = [31, 51, 71]
if min(ema_spans) < 2:
    raise ValueError("EMA span must be >= 2.")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, ema_span in zip(axes, ema_spans):
    upstream_ema = df["Upstream Pressure"].ewm(span=ema_span, adjust=False).mean()
    downstream_ema = df["Downstream Pressure"].ewm(span=ema_span, adjust=False).mean()

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], upstream_ema, label=f"Upstream EMA (span={ema_span})")
    ax.plot(df["Time"], downstream_ema, label=f"Downstream EMA (span={ema_span})")
    ax.set_ylabel("Pressure")
    ax.set_title(f"Upstream vs Downstream Pressure (EMA span={ema_span})")
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()

## Low-Pass Filter

In [ ]:
# Digital low-pass filter visualization (Butterworth)
from scipy.signal import butter, filtfilt

cutoffs = [0.2, 0.1, 0.05]
filter_order = 3
if any(c <= 0 or c >= 0.5 for c in cutoffs):
    raise ValueError("Cutoff values must be between 0 and 0.5 (Nyquist normalized).")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, cutoff in zip(axes, cutoffs):
    b, a = butter(filter_order, cutoff, btype="low", analog=False)
    upstream_lp = filtfilt(b, a, df["Upstream Pressure"].to_numpy())
    downstream_lp = filtfilt(b, a, df["Downstream Pressure"].to_numpy())

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], upstream_lp, label=f"Upstream LPF (cutoff={cutoff})")
    ax.plot(df["Time"], downstream_lp, label=f"Downstream LPF (cutoff={cutoff})")
    ax.set_ylabel("Pressure")
    ax.set_title(f"Low-Pass Filter (order={filter_order}, cutoff={cutoff})")
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()

In [ ]:
# Median filter visualization
from scipy.signal import medfilt

window_sizes = [5, 11, 21]
if any(w < 3 for w in window_sizes):
    raise ValueError("Median filter window size must be >= 3.")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, window_size in zip(axes, window_sizes):
    if window_size % 2 == 0:
        window_size += 1

    upstream_med = medfilt(df["Upstream Pressure"].to_numpy(), kernel_size=window_size)
    downstream_med = medfilt(df["Downstream Pressure"].to_numpy(), kernel_size=window_size)

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], upstream_med, label=f"Upstream Median (window={window_size})")
    ax.plot(df["Time"], downstream_med, label=f"Downstream Median (window={window_size})")
    ax.set_ylabel("Pressure")
    ax.set_title(f"Median Filter (window={window_size})")
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()

In [ ]:
# Hampel filter visualization
import numpy as np

def hampel_filter(series, window_size, n_sigmas=3.0):
    if window_size % 2 == 0:
        window_size += 1
    k = window_size // 2
    values = series.to_numpy()
    filtered = values.copy()
    for i in range(k, len(values) - k):
        window = values[i - k : i + k + 1]
        median = np.median(window)
        mad = np.median(np.abs(window - median))
        if mad == 0:
            continue
        if np.abs(values[i] - median) > n_sigmas * 1.4826 * mad:
            filtered[i] = median
    return filtered

window_sizes = [7, 15, 31]
n_sigmas = 3.0
if any(w < 3 for w in window_sizes):
    raise ValueError("Hampel window size must be >= 3.")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, window_size in zip(axes, window_sizes):
    upstream_hampel = hampel_filter(df["Upstream Pressure"], window_size, n_sigmas=n_sigmas)
    downstream_hampel = hampel_filter(df["Downstream Pressure"], window_size, n_sigmas=n_sigmas)

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], upstream_hampel, label=f"Upstream Hampel (window={window_size})")
    ax.plot(df["Time"], downstream_hampel, label=f"Downstream Hampel (window={window_size})")
    ax.set_ylabel("Pressure")
    ax.set_title(f"Hampel Filter (window={window_size}, n_sigmas={n_sigmas})")
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()

In [ ]:
# Hampel + low-pass filter visualization
from scipy.signal import butter, filtfilt

combined_window_sizes = [7, 15, 31]
combined_cutoffs = [0.2, 0.1, 0.05]
combined_order = 3
combined_n_sigmas = 3.0

if len(combined_window_sizes) != len(combined_cutoffs):
    raise ValueError("Combined filter settings must have the same length.")
if any(w < 3 for w in combined_window_sizes):
    raise ValueError("Hampel window size must be >= 3.")
if any(c <= 0 or c >= 0.5 for c in combined_cutoffs):
    raise ValueError("Cutoff values must be between 0 and 0.5 (Nyquist normalized).")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, window_size, cutoff in zip(axes, combined_window_sizes, combined_cutoffs):
    upstream_hampel = hampel_filter(df["Upstream Pressure"], window_size, n_sigmas=combined_n_sigmas)
    downstream_hampel = hampel_filter(df["Downstream Pressure"], window_size, n_sigmas=combined_n_sigmas)

    b, a = butter(combined_order, cutoff, btype="low", analog=False)
    upstream_combo = filtfilt(b, a, upstream_hampel)
    downstream_combo = filtfilt(b, a, downstream_hampel)

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(
        df["Time"],
        upstream_combo,
        label=f"Upstream Hampel+LPF (window={window_size}, cutoff={cutoff})",
    )
    ax.plot(
        df["Time"],
        downstream_combo,
        label=f"Downstream Hampel+LPF (window={window_size}, cutoff={cutoff})",
    )
    ax.set_ylabel("Pressure")
    ax.set_title(
        "Hampel + Low-Pass Filter "
        f"(window={window_size}, n_sigmas={combined_n_sigmas}, cutoff={cutoff})"
    )
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()